In [1]:
%%bash

project_base_path=/home/liuzhiyu/Projects/neo_caste/anno_cells/
ln -sf $project_base_path/Acer/matrix/01_Acer_transfromHY.h5ad ./matrix

In [2]:
import scanpy as sc

In [3]:
Acer = sc.read_h5ad("./matrix/01_Acer_transfromHY.h5ad")
Acer

AnnData object with n_obs × n_vars = 58166 × 11817
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'batch', 'leiden_res0.25', 'leiden_res0.50', 'leiden_res1.00', 'HY_barcode', 'celltype_transfrom_HY'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'batch_colors', 'celltype_transfrom_HY_colors', 'hvg', 'leiden_res0.25', 'leiden_res0.25_colors', 'leiden_res0.50', 'leiden_res0.50_colors', 'leiden_res1.00', 'leiden_res1.00_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_scVI', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'logcounts'
    obsp: 'connectivities', 'distances'

In [7]:
import ClustAssessPy as ca

In [ ]:
adj = Acer.obsp['connectivities']
resolutions = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00, 1.10, 1.20]

stab = ca.assess_clustering_stability(
    adj,
    resolution=resolutions,
    ncores=16, # 按机器核数调
    n_repetitions=100, # 每个 resolution 换 100 个种子
    ecs_thresh=1,
    algorithms=['leiden'],
    flavor='default', # 即 leidenalg
)


---

In [9]:
import pickle
import numpy as np
import matplotlib
# matplotlib.use("Agg")          # 无显示器/tmux 下必须
import pandas as pd
import matplotlib.pyplot as plt
import ClustAssessPy as ca

In [ ]:
# 1) 保存完整结果，供之后重新画图
with open("./metadata/clust_stability.pkl", "wb") as f:
    pickle.dump(stab, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# 2) res 与 k 对应图
fig = ca.plot_k_resolution_corresp(stab)
fig.savefig("./figures/res_vs_k.png", dpi=300, bbox_inches="tight")

In [ ]:
# 3) ECC 随 res 的表格（推荐自己算，原因见下）
rows = []
for method, res_dict in stab["split_by_resolution"].items():
    for res, obj in res_dict.items():
        ecc = np.asarray(obj["ecc"], dtype=float)
        k_freq = {int(k): sum(p["freq"] for p in v["partitions"])
                for k, v in obj["clusters"].items()}
        rows.append({
            "method": method,
            "resolution": float(res),
            "k_mode": max(k_freq, key=k_freq.get),
            "ecc_median": np.median(ecc),
            "ecc_min": ecc.min(),
            "ecc_max": ecc.max(),
        })
df = pd.DataFrame(rows).sort_values(["method", "resolution"])
df.to_csv("./metadata/res_stability_summary.csv", index=False)
print(df)